# 03. Calidad de datos y análisis exploratorio

## Objetivo

Auditar y preparar el dataset maestro para el desarrollo de modelos de predicción
de incidencia de enfermedad hepática a 10 años.

En este notebook se analizarán:

- Integridad y estructura del dataset.
- Fuga temporal y fuga de información.
- Distribución de la variable objetivo.
- Valores ausentes.
- Códigos especiales de no respuesta.
- Variables constantes o cuasi-constantes.
- Outliers y valores clínicamente imposibles.
- Distribuciones y relaciones preliminares.
- Variables redundantes y correlacionadas.

Las transformaciones que dependen de los datos, como imputación, escalado,
balanceo o PCA, se ajustarán posteriormente utilizando únicamente el conjunto
de entrenamiento.

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42
TARGET = "incident_liver_disease_10y"
DATASET_NAME = "klosa_liver_incident_10y_master"

In [2]:
def find_project_root(start_path: Path) -> Path:
    """
    Busca la raíz del proyecto identificando una carpeta que contenga
    simultáneamente los directorios 'data' y 'notebooks'.
    """
    start_path = start_path.resolve()

    candidate_paths = [start_path, *start_path.parents]

    for candidate in candidate_paths:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la raíz del proyecto. "
        "Comprueba que existen las carpetas 'data' y 'notebooks'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "data_quality"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de datos procesados: {PROCESSED_DIR}")
print(f"Directorio de informes: {REPORTS_DIR}")

Raíz del proyecto: C:\Users\DAVID\TFM_Liver_Disease_Risk
Directorio de datos procesados: C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed
Directorio de informes: C:\Users\DAVID\TFM_Liver_Disease_Risk\reports\data_quality


In [3]:
SUPPORTED_EXTENSIONS = [".csv", ".xlsx", ".xls", ".parquet"]

dataset_candidates = []

for extension in SUPPORTED_EXTENSIONS:
    dataset_candidates.extend(
        DATA_DIR.rglob(f"{DATASET_NAME}{extension}")
    )

dataset_candidates = sorted(set(dataset_candidates))

if not dataset_candidates:
    raise FileNotFoundError(
        f"No se ha encontrado '{DATASET_NAME}' dentro de {DATA_DIR}"
    )

print("Archivos encontrados:")

for index, path in enumerate(dataset_candidates, start=1):
    print(f"{index}. {path}")

DATASET_PATH = dataset_candidates[0]

print(f"\nDataset seleccionado: {DATASET_PATH}")

Archivos encontrados:
1. C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_incident_10y_master.csv
2. C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_incident_10y_master.parquet

Dataset seleccionado: C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_incident_10y_master.csv


In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    """Carga un dataset según su extensión."""

    suffix = path.suffix.lower()

    if suffix == ".csv":
        try:
            return pd.read_csv(path, encoding="utf-8", low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="latin-1", low_memory=False)

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Formato no soportado: {suffix}")


df = load_dataset(DATASET_PATH)
df_raw = df.copy(deep=True)

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

Dataset cargado correctamente.
Filas: 6,560
Columnas: 310


In [5]:
if TARGET not in df.columns:
    target_candidates = [
        column
        for column in df.columns
        if "liver" in column.lower()
        or "incident" in column.lower()
        or "target" in column.lower()
    ]

    raise KeyError(
        f"No se ha encontrado la variable objetivo '{TARGET}'.\n"
        f"Posibles candidatas: {target_candidates}"
    )

print(f"Variable objetivo encontrada: {TARGET}")

target_values = sorted(
    df[TARGET].dropna().unique().tolist()
)

print(f"Valores encontrados: {target_values}")

if not set(target_values).issubset({0, 1}):
    print(
        "ADVERTENCIA: la variable objetivo contiene valores diferentes de 0 y 1."
    )

Variable objetivo encontrada: incident_liver_disease_10y
Valores encontrados: [0, 1]


1. Comprobamos filas duplicadas que puedan interferir en la calidad de los datos


In [6]:
n_duplicate_rows = df.duplicated().sum()

print(f"Filas completamente duplicadas: {n_duplicate_rows:,}")
print(f"Porcentaje: {n_duplicate_rows / len(df) * 100:.3f}%")

Filas completamente duplicadas: 0
Porcentaje: 0.000%


2. Detectamos variables cuasi-constantes, donde un único valor representa más del 99 %:

In [7]:
unique_counts = df.nunique(dropna=False).sort_values()

constant_columns = unique_counts[
    unique_counts <= 1
].index.tolist()

print(f"Variables constantes: {len(constant_columns)}")
print(constant_columns)

Variables constantes: 2
['mniw_y', 'g013']


In [8]:
quasi_constant_report = []

for column in df.columns:
    frequencies = df[column].value_counts(
        normalize=True,
        dropna=False
    )

    if len(frequencies) == 0:
        continue

    dominant_value = frequencies.index[0]
    dominant_frequency = frequencies.iloc[0]

    if dominant_frequency >= 0.99:
        quasi_constant_report.append({
            "variable": column,
            "valor_dominante": dominant_value,
            "porcentaje_dominante": dominant_frequency * 100,
            "valores_unicos": df[column].nunique(dropna=False)
        })

quasi_constant_report = pd.DataFrame(
    quasi_constant_report
).sort_values(
    "porcentaje_dominante",
    ascending=False
)

display(quasi_constant_report)

,variable,valor_dominante,porcentaje_dominante,valores_unicos
21,g013,NaN,100.000,1
24,mniw_y,"2,006.000",100.000,1
1,a035_07,NaN,99.985,2
22,g014,NaN,99.985,2
23,industrial,NaN,99.939,4
4,businessfarm,NaN,99.909,7
27,unemployment,NaN,99.817,11
26,personal,NaN,99.771,16
17,d_com108,NaN,99.695,7
20,g012,NaN,99.466,10


3. Buscaremos nombres que parezcan pertenecer a olas posteriores:

In [9]:
future_wave_patterns = [
    r"(^|_)w0?[2-9]($|_)",
    r"wave[_ ]?[2-9]",
    r"ola[_ ]?[2-9]",
    r"follow.?up",
    r"incident",
    r"event",
    r"diagnos",
    r"liver",
    r"hepatic",
    r"onset",
    r"develop"
]

potential_leakage_columns = []

for column in df.columns:
    column_lower = column.lower()

    matched_patterns = [
        pattern
        for pattern in future_wave_patterns
        if re.search(pattern, column_lower)
    ]

    if matched_patterns:
        potential_leakage_columns.append({
            "variable": column,
            "patrones_detectados": ", ".join(matched_patterns),
            "valores_unicos": df[column].nunique(dropna=True),
            "porcentaje_nulos": df[column].isna().mean() * 100
        })

potential_leakage_report = pd.DataFrame(
    potential_leakage_columns
).sort_values("variable")

display(potential_leakage_report)

,variable,patrones_detectados,valores_unicos,porcentaje_nulos
0,incident_liver_disease_10y,"incident, liver",2,0.000


4. Balance de clases

In [10]:
target_distribution = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis("clase")
    .reset_index(name="frecuencia")
)

target_distribution["porcentaje"] = (
    target_distribution["frecuencia"] / len(df) * 100
)

display(target_distribution)

,clase,frecuencia,porcentaje
0,0,6429,98.003
1,1,131,1.997


5. Valores perdidos (missing values)

In [12]:
missing_summary = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje_nulos": df.isna().mean().mul(100),
    "tipo": df.dtypes.astype(str),
    "valores_unicos": df.nunique(dropna=True)
})

missing_summary = (
    missing_summary
    .query("nulos > 0")
    .sort_values("porcentaje_nulos", ascending=False)
)

display(missing_summary)

,nulos,porcentaje_nulos,tipo,valores_unicos
g013,6560,100.000,float64,0
g014,6559,99.985,float64,1
a035_07,6559,99.985,float64,1
industrial,6556,99.939,float64,3
businessfarm,6554,99.909,float64,6
...,...,...,...,...
c145,42,0.640,float64,4
c142,42,0.640,float64,4
c149,42,0.640,float64,4
c081,20,0.305,float64,2


In [13]:
missing_bands = pd.cut(
    df.isna().mean().mul(100),
    bins=[-0.01, 0, 5, 20, 40, 60, 80, 100],
    labels=[
        "0%",
        "0-5%",
        "5-20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "80-100%"
    ]
)

display(
    missing_bands
    .value_counts()
    .sort_index()
    .rename("numero_variables")
    .to_frame()
)

,numero_variables
0%,93
0-5%,14
5-20%,7
20-40%,6
40-60%,40
60-80%,28
80-100%,122


6. Comprobación de ausencia entre clases

In [14]:
target_classes = sorted(df[TARGET].dropna().unique())

missing_by_target = pd.DataFrame({
    f"missing_clase_{target_class}": (
        df.loc[df[TARGET] == target_class]
        .isna()
        .mean()
        .mul(100)
    )
    for target_class in target_classes
})

if len(target_classes) == 2:
    class_0 = target_classes[0]
    class_1 = target_classes[1]

    missing_by_target["diferencia_absoluta"] = (
        missing_by_target[f"missing_clase_{class_1}"]
        - missing_by_target[f"missing_clase_{class_0}"]
    ).abs()

    missing_by_target = missing_by_target.sort_values(
        "diferencia_absoluta",
        ascending=False
    )

display(missing_by_target.head(30))

,missing_clase_0,missing_clase_1,diferencia_absoluta
chronic_j,58.345,41.985,16.360
c068,41.655,58.015,16.360
labor_st,47.146,59.542,12.396
retired,47.146,59.542,12.396
residence,47.955,36.641,11.313
c312,62.918,73.282,10.364
c311,62.918,73.282,10.364
residence_,20.314,30.534,10.220
a035_02,41.002,51.145,10.143
c303,41.873,51.908,10.036


7. Detección de códigos espceiales

In [15]:
suspect_codes = [
    -999999, -99999, -9999, -999, -99, -9,
    -8, -7, -1,
    97, 98, 99,
    997, 998, 999,
    9997, 9998, 9999,
    99997, 99998, 99999,
    999997, 999998, 999999
]

special_code_report = []

numeric_columns = df.select_dtypes(
    include=np.number
).columns

for column in numeric_columns:
    value_counts = df[column].value_counts(dropna=False)

    for code in suspect_codes:
        if code in value_counts.index:
            count = int(value_counts.loc[code])

            special_code_report.append({
                "variable": column,
                "codigo_sospechoso": code,
                "frecuencia": count,
                "porcentaje": count / len(df) * 100
            })

special_code_report = pd.DataFrame(special_code_report)

if not special_code_report.empty:
    special_code_report = special_code_report.sort_values(
        ["variable", "codigo_sospechoso"]
    )

display(special_code_report)

,variable,codigo_sospechoso,frecuencia,porcentaje
0,a002_age,97,1,0.015
1,a030,97,55,0.838
2,agriculture,-9,15,0.229
3,agriculture,-8,4,0.061
4,annuity,-8,3,0.046
5,annuity,99,1,0.015
6,assetinc,-9,156,2.378
7,ba068,-9,6,0.091
8,ba068,-8,2,0.030
9,businessfarm,-9,1,0.015


In [16]:
feature_columns = [
    column
    for column in df.columns
    if column != TARGET
]

numeric_features = (
    df[feature_columns]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

non_numeric_features = [
    column
    for column in feature_columns
    if column not in numeric_features
]

low_cardinality_numeric = [
    column
    for column in numeric_features
    if df[column].nunique(dropna=True) <= 15
]

continuous_numeric_candidates = [
    column
    for column in numeric_features
    if df[column].nunique(dropna=True) > 15
]

print(f"Variables predictoras totales: {len(feature_columns)}")
print(f"Variables numéricas: {len(numeric_features)}")
print(f"Variables no numéricas: {len(non_numeric_features)}")
print(
    "Variables numéricas posiblemente categóricas: "
    f"{len(low_cardinality_numeric)}"
)
print(
    "Variables numéricas posiblemente continuas: "
    f"{len(continuous_numeric_candidates)}"
)

Variables predictoras totales: 309
Variables numéricas: 309
Variables no numéricas: 0
Variables numéricas posiblemente categóricas: 221
Variables numéricas posiblemente continuas: 88


In [17]:
categorical_candidate_report = pd.DataFrame({
    "variable": low_cardinality_numeric,
    "valores_unicos": [
        df[column].nunique(dropna=True)
        for column in low_cardinality_numeric
    ],
    "valores": [
        sorted(df[column].dropna().unique().tolist())
        for column in low_cardinality_numeric
    ]
})

display(categorical_candidate_report)

,variable,valores_unicos,valores
0,a002m,12,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
1,a003,2,"[1, 5]"
2,a030,6,"[1, 2, 3, 4, 5, 97]"
3,a032,10,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
4,a035_01,10,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]"
...,...,...,...
216,size_c,11,"[-9.0, -8.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]"
217,smoke,3,"[0, 1, 2]"
218,status,6,"[-8.0, 1.0, 3.0, 5.0, 7.0, 9.0]"
219,target1,2,"[1.0, 2.0]"


In [19]:
summary_initial = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": df.isna().mean().mul(100),
    "valores_unicos": df.nunique(dropna=True)
}).sort_values(
    ["porcentaje_nulos", "valores_unicos"],
    ascending=[False, True]
)

display(summary_initial)

,tipo,no_nulos,nulos,porcentaje_nulos,valores_unicos
g013,float64,0,6560,100.000,0
a035_07,float64,1,6559,99.985,1
g014,float64,1,6559,99.985,1
industrial,float64,4,6556,99.939,3
businessfarm,float64,6,6554,99.909,6
...,...,...,...,...,...
year2,int64,6560,0,0.000,62
c105,int64,6560,0,0.000,70
hhinc,int64,6560,0,0.000,263
wgt_c,float64,6560,0,0.000,930


In [20]:
summary_initial.to_csv(
    REPORTS_DIR / "initial_variable_summary.csv",
    encoding="utf-8-sig"
)

missing_summary.to_csv(
    REPORTS_DIR / "missing_values_summary.csv",
    encoding="utf-8-sig"
)

quasi_constant_report.to_csv(
    REPORTS_DIR / "quasi_constant_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

potential_leakage_report.to_csv(
    REPORTS_DIR / "potential_data_leakage.csv",
    index=False,
    encoding="utf-8-sig"
)

special_code_report.to_csv(
    REPORTS_DIR / "special_codes_report.csv",
    index=False,
    encoding="utf-8-sig"
)

target_distribution.to_csv(
    REPORTS_DIR / "target_distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Informes guardados en: {REPORTS_DIR}")

Informes guardados en: C:\Users\DAVID\TFM_Liver_Disease_Risk\reports\data_quality


['pid',
 'a002_age',
 'a002m',
 'a002y',
 'a003',
 'a030',
 'a032',
 'a035_01',
 'a035_02',
 'a035_03',
 'a035_04',
 'a035_05',
 'a035_06',
 'a035_07',
 'addic',
 'adl',
 'agriculture',
 'alc',
 'annuity',
 'assetinc',
 'ba003',
 'ba068',
 'ba075',
 'ba_resp',
 'bb_adl1',
 'bb_adl2',
 'bb_adl3',
 'bb_adl_num1',
 'bb_adl_num2',
 'bb_adl_num3',
 'bm3',
 'bm4',
 'bm5',
 'bm6',
 'bmi',
 'bp1',
 'bp1_1',
 'bp1_2',
 'bp3',
 'bp4',
 'bp5',
 'bp6',
 'businessfarm',
 'c001',
 'c003',
 'c005',
 'c007m',
 'c007y',
 'c012m',
 'c012y',
 'c017m',
 'c017y',
 'c024m',
 'c024y',
 'c034m',
 'c034y',
 'c039m',
 'c039y',
 'c044m',
 'c044y',
 'c049m',
 'c049y',
 'c056',
 'c064m',
 'c064y',
 'c068',
 'c081',
 'c082',
 'c085',
 'c102',
 'c105',
 'c106',
 'c107',
 'c108',
 'c109',
 'c111',
 'c112',
 'c124m',
 'c124y',
 'c126',
 'c127',
 'c128',
 'c129',
 'c130',
 'c131',
 'c132',
 'c133',
 'c134',
 'c135',
 'c142',
 'c143',
 'c144',
 'c145',
 'c146',
 'c147',
 'c148',
 'c149',
 'c150',
 'c151',
 'c152',
 'c20